In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.special import erf
from scipy.spatial.distance import cdist
from scipy.linalg import norm
from scipy.sparse.linalg import gmres, LinearOperator, spilu
from scipy.sparse import csc_matrix, identity, block_diag
from numpy.polynomial.legendre import leggauss
from time import time
from scipy.interpolate import griddata
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import minimize_scalar
import scipy.linalg as la

In [ ]:
def rbf_tension(r, tau):
    return -1.0 / (2.0 * tau**3) * (np.exp(-tau * r) + tau * r)

def rbf_tension_derivative(r, tau=1.0):
    return -1.0 / (2.0 * tau**2) * (1.0 - np.exp(-tau * r))

def laplacian_rbf_tension_2d(r, tau):
    r = np.asarray(r, dtype=float)   
    exp_term = np.exp(-tau * r)
    out = np.empty_like(r)
    mask = (r != 0)
    out[mask] = -0.5/tau * exp_term[mask] - 0.5/(tau**2 * r[mask]) * (1.0 - exp_term[mask])
    out[~mask] = -1.0 / tau
    return out

In [ ]:
def polynomial_basis(x, y, deg):
    if np.isscalar(x):
        x, y = np.array([x]), np.array([y])
    basis = [x**i * y**j for Td in range(deg + 1) for i in range(Td + 1) for j in [Td - i]]
    return np.array(basis).T

def polynomial_laplacian(x, y, poly_idx, deg):
    count = 0
    for Td in range(deg + 1):
        for i in range(Td + 1):
            j = Td - i
            if count == poly_idx:
                lap = 0
                if i >= 2: lap += i*(i-1)*x**(i-2)*y**j
                if j >= 2: lap += j*(j-1)*x**i*y**(j-2)
                return lap
            count += 1
    return 0

def polynomial_gradients(x, y, deg):
    gradients = []
    for Td in range(deg + 1):
        for i in range(Td + 1):
            j = Td - i
            grad_x = i * x**(max(i-1, 0)) * y**j if i > 0 else 0
            grad_y = j * x**i * y**(max(j-1, 0)) if j > 0 else 0
            gradients.append([grad_x, grad_y])
    return np.array(gradients)

In [ ]:
def halton_sequence(n, base, skip=10):
    seq = np.zeros(n)
    for i in range(n):
        f = 1.0
        r = 0.0
        idx = i + skip + 1  
        while idx > 0:
            f = f / base
            r = r + f * (idx % base)
            idx = idx // base
        seq[i] = r
    return seq

def flower_radius(theta, n_petals=5, a=0.3, b=0.15):
    return a + b * np.cos(n_petals * theta)

def is_point_in_flower(x, y, n_petals=5, a=0.3, b=0.15):
    x_c, y_c = x - 0.5, y - 0.5
    r = np.sqrt(x_c**2 + y_c**2)
    theta = np.arctan2(y_c, x_c)
    return r <= flower_radius(theta, n_petals, a, b)

def generate_flower_domain_points(n_interior_target, n_boundary_target, n_petals=5, a=0.3, b=0.15):
    interior_points = []
    n_candidates = n_interior_target * 6  
    halton_x = halton_sequence(n_candidates, 2)
    halton_y = halton_sequence(n_candidates, 3)  
    for i in range(n_candidates):
        x, y = halton_x[i], halton_y[i]
        if is_point_in_flower(x, y, n_petals, a, b):
            x_c, y_c = x - 0.5, y - 0.5
            r = np.sqrt(x_c**2 + y_c**2)
            theta = np.arctan2(y_c, x_c)
            if flower_radius(theta, n_petals, a, b) - r >= 0.001:  
                interior_points.append([x, y])
                if len(interior_points) >= n_interior_target:
                    break   

    halton_t = halton_sequence(n_boundary_target, 5)
    theta_points = 2 * np.pi * halton_t
    boundary_points = []
    for theta in theta_points:
        r = flower_radius(theta, n_petals, a, b)
        x = 0.5 + r * np.cos(theta)
        y = 0.5 + r * np.sin(theta)
        boundary_points.append([x, y])        
    
    return np.array(interior_points[:n_interior_target]), np.array(boundary_points)

In [ ]:
def build_matrices(interior_points, boundary_points, tau, polydeg):
    all_points = np.vstack([interior_points, boundary_points])
    n, n_prime = len(interior_points), len(boundary_points)
    dm = (polydeg + 1) * (polydeg + 2) // 2 
    dists = cdist(all_points, all_points)
    K = rbf_tension(dists, tau)
    Q = polynomial_basis(all_points[:, 0], all_points[:, 1], polydeg) 
    K1, K2, M = K[:n, :n], K[n:, n:], K[:n, n:]
    Q1, Q2 = Q[:n, :], Q[n:, :]
    A_mat = np.block([[K2, Q2], [Q2.T, np.zeros((dm, dm))]])
    A_inv = np.linalg.inv(A_mat)
    MQ1 = np.hstack([M, Q1])
    B = MQ1 @ A_inv
    S = K1 - MQ1 @ A_inv @ MQ1.T
    return {'S': S, 'B': B, 'A_inv': A_inv, 'n': n, 'n_prime': n_prime, 'dm': dm}


def compute_basis_laplacians(interior_points, boundary_points, matrices, tau, polydeg):
    n, n_prime, dm = matrices['n'], matrices['n_prime'], matrices['dm']
    La, Lb = np.zeros((n, n)), np.zeros((n, n_prime + dm))
    S_inv = np.linalg.inv(matrices['S'])
    C = matrices['A_inv'] + matrices['B'].T @ S_inv @ matrices['B']   
    for i in range(n):
        r_int = np.linalg.norm(interior_points[i] - interior_points, axis=1)
        r_bnd = np.linalg.norm(interior_points[i] - boundary_points, axis=1)
        lap_lambda = laplacian_rbf_tension_2d(r_int, tau)
        lap_gamma = laplacian_rbf_tension_2d(r_bnd, tau)
        lap_theta = np.array([polynomial_laplacian(interior_points[i, 0], interior_points[i, 1], j, polydeg) for j in range(dm)])
        lap_gamma_theta = np.concatenate([lap_gamma, lap_theta])
        La[i, :] = -S_inv @ (matrices['B'] @ lap_gamma_theta - lap_lambda)
        Lb[i, :] = C @ lap_gamma_theta - matrices['B'].T @ S_inv @ lap_lambda
    return La, Lb


def compute_basis_gradients(eval_points, interior_points, boundary_points, matrices, tau=1.0, polydeg=1):
    n_eval, n, n_prime, dm = len(eval_points), matrices['n'], matrices['n_prime'], matrices['dm']
    grad_a, grad_b = np.zeros((n_eval, n, 2)), np.zeros((n_eval, n_prime + dm, 2))
    S_inv = np.linalg.inv(matrices['S'])
    C = matrices['A_inv'] + matrices['B'].T @ S_inv @ matrices['B']
    for idx, x_point in enumerate(eval_points):
        dx_int = x_point - interior_points
        dx_bnd = x_point - boundary_points
        r_int = np.maximum(np.linalg.norm(dx_int, axis=1), 1e-12)
        r_bnd = np.maximum(np.linalg.norm(dx_bnd, axis=1), 1e-12)   
        phi_prime_int = rbf_tension_derivative(r_int, tau)
        phi_prime_bnd = rbf_tension_derivative(r_bnd, tau)
        grad_lambda = (phi_prime_int[:, np.newaxis] / r_int[:, np.newaxis]) * dx_int
        grad_gamma = (phi_prime_bnd[:, np.newaxis] / r_bnd[:, np.newaxis]) * dx_bnd 
        grad_poly = polynomial_gradients(x_point[0], x_point[1], polydeg)
        grad_gamma_theta = np.vstack([grad_gamma, grad_poly])
        S_inv_grad_lambda = S_inv @ grad_lambda
        grad_a[idx] = -S_inv @ (matrices['B'] @ grad_gamma_theta) + S_inv_grad_lambda
        grad_b[idx] = C @ grad_gamma_theta - matrices['B'].T @ S_inv_grad_lambda
    return grad_a, grad_b

In [ ]:
def evaluate_basis_functions(points, interior_points, boundary_points, matrices, tau=1.0, polydeg=1):
    n, n_prime, dm = matrices['n'], matrices['n_prime'], matrices['dm']
    S_inv = np.linalg.inv(matrices['S'])
    C = matrices['A_inv'] + matrices['B'].T @ S_inv @ matrices['B']
    dists_int = cdist(points, interior_points)
    dists_bnd = cdist(points, boundary_points)
    lambda_mat = rbf_tension(dists_int, tau)
    gamma_mat = rbf_tension(dists_bnd, tau)
    theta_mat = polynomial_basis(points[:, 0], points[:, 1], polydeg)
    gamma_theta_mat = np.hstack([gamma_mat, theta_mat])
    a_mat = (lambda_mat - gamma_theta_mat @ matrices['B'].T) @ S_inv
    b_mat = gamma_theta_mat @ C - (lambda_mat @ S_inv) @ matrices['B']
    return a_mat, b_mat

def compute_Ka_Kb_matrices(interior_points, boundary_points, matrices, quad_points, quad_weights, domain_checker, tau=1.0, alpha=2.0, polydeg=1):
    indicator = np.array([domain_checker(p[0], p[1]) for p in quad_points], dtype=float)
    a_quad, b_quad = evaluate_basis_functions(quad_points, interior_points, boundary_points, matrices, tau, polydeg)
    n, n_bdr_poly = matrices['n'], matrices['n_prime'] + matrices['dm']
    Ka, Kb = np.zeros((n, n)), np.zeros((n, n_bdr_poly)) 
    for i in range(n):
        diff = interior_points[i] - quad_points
        dist_sq = np.sum(diff**2, axis=1)
        k_vals = np.exp(-alpha * dist_sq) * indicator  
        Ka[i, :] = np.sum(quad_weights[:, np.newaxis] * k_vals[:, np.newaxis] * a_quad, axis=0)
        Kb[i, :] = np.sum(quad_weights[:, np.newaxis] * k_vals[:, np.newaxis] * b_quad, axis=0)
    return Ka, Kb

def compute_Ka_Kb_gradients(interior_points, boundary_points, matrices, quad_points, quad_weights, domain_checker, tau=1.0, alpha=2.0, polydeg=1):
    indicator = np.array([domain_checker(p[0], p[1]) for p in quad_points], dtype=float)
    a_quad, b_quad = evaluate_basis_functions(quad_points, interior_points, boundary_points, matrices, tau, polydeg)
    n, n_bdr_poly = matrices['n'], matrices['n_prime'] + matrices['dm']
    dKa_dx, dKa_dy = np.zeros((n, n)), np.zeros((n, n))
    dKb_dx, dKb_dy = np.zeros((n, n_bdr_poly)), np.zeros((n, n_bdr_poly))
    
    for i in range(n):
        diff = interior_points[i] - quad_points
        dist_sq = np.sum(diff**2, axis=1)
        k_vals = np.exp(-alpha * dist_sq) * indicator  # Apply indicator
        factor = -2 * alpha
        dk_dx = factor * diff[:, 0] * k_vals  
        dk_dy = factor * diff[:, 1] * k_vals
        dKa_dx[i, :] = np.sum(quad_weights[:, np.newaxis] * dk_dx[:, np.newaxis] * a_quad, axis=0)
        dKa_dy[i, :] = np.sum(quad_weights[:, np.newaxis] * dk_dy[:, np.newaxis] * a_quad, axis=0)
        dKb_dx[i, :] = np.sum(quad_weights[:, np.newaxis] * dk_dx[:, np.newaxis] * b_quad, axis=0)
        dKb_dy[i, :] = np.sum(quad_weights[:, np.newaxis] * dk_dy[:, np.newaxis] * b_quad, axis=0)
    return dKa_dx, dKa_dy, dKb_dx, dKb_dy

In [ ]:
def precompute_manufactured_solution(times, interior_points, alpha, n_petals=5, a=0.3, b=0.15):
    print("   Precomputing manufactured solution integrals...")
    start_time = time()
    n_times = len(times)
    n_points = len(interior_points)
    order = 40  
    quad_points, quad_weights = setup_quadrature(order=order, a_int=0.0, b_int=1.0)
    indicator = np.array([is_point_in_flower(p[0], p[1], n_petals, a, b) for p in quad_points], dtype=float)
    W_u_all = np.zeros((n_times, n_points, 2))
    grad_W_all = np.zeros((n_times, n_points, 2, 2))
    for t_idx, t in enumerate(times):
        growth_factor_1 = 1 + np.tanh(0.5*t)
        growth_factor_2 = 1 + 0.5*np.tanh(0.3*t)
        u1_quad = growth_factor_1 * np.sin(np.pi * quad_points[:, 0]) * np.sin(np.pi * quad_points[:, 1]) * indicator
        u2_quad = growth_factor_2 * np.cos(np.pi * quad_points[:, 0]) * np.cos(np.pi * quad_points[:, 1]) * indicator 
        for i, point in enumerate(interior_points):
            diff_x = point[0] - quad_points[:, 0]
            diff_y = point[1] - quad_points[:, 1]
            dist_sq = diff_x**2 + diff_y**2
            k_vals = np.exp(-alpha * dist_sq) * indicator  
            dk_dx = -2 * alpha * diff_x * k_vals  
            dk_dy = -2 * alpha * diff_y * k_vals
            W_u_all[t_idx, i, 0] = np.sum(quad_weights * k_vals * u1_quad)
            W_u_all[t_idx, i, 1] = np.sum(quad_weights * k_vals * u2_quad)
            grad_W_all[t_idx, i, 0, 0] = np.sum(quad_weights * dk_dx * u1_quad)
            grad_W_all[t_idx, i, 1, 0] = np.sum(quad_weights * dk_dy * u1_quad)
            grad_W_all[t_idx, i, 0, 1] = np.sum(quad_weights * dk_dx * u2_quad)
            grad_W_all[t_idx, i, 1, 1] = np.sum(quad_weights * dk_dy * u2_quad) 
    print(f"   ...precomputation completed in {time() - start_time:.2f}s")
    return W_u_all, grad_W_all

In [ ]:
def u_exact_general(t, x, y):
    u = np.zeros((2,) + np.shape(x))
    growth_factor_1 = 1 + np.tanh(0.5*t)
    growth_factor_2 = 1 + 0.5*np.tanh(0.3*t) 
    u[0] = growth_factor_1 * np.sin(np.pi * x) * np.sin(np.pi * y)
    u[1] = growth_factor_2 * np.cos(np.pi * x) * np.cos(np.pi * y)
    return u

def u_exact_derivatives_general(t, x, y):
    pi = np.pi
    tanh_1 = np.tanh(0.5*t)
    tanh_2 = np.tanh(0.3*t)
    growth_factor_1 = 1 + tanh_1
    growth_factor_2 = 1 + 0.5*tanh_2
    dgrowth_dt_1 = 0.5 * (1 - tanh_1**2)  
    dgrowth_dt_2 = 0.15 * (1 - tanh_2**2)
    spatial_1 = np.sin(pi * x) * np.sin(pi * y)
    spatial_2 = np.cos(pi * x) * np.cos(pi * y)
    u_vals = u_exact_general(t, x, y)

    u_t = np.zeros((2,) + np.shape(x))
    u_t[0] = dgrowth_dt_1 * spatial_1
    u_t[1] = dgrowth_dt_2 * spatial_2
    grad_u = np.zeros((2, 2, *np.shape(x)))
    grad_u[0, 0] = growth_factor_1 * pi * np.cos(pi * x) * np.sin(pi * y)  # ∂u₁/∂x
    grad_u[0, 1] = growth_factor_1 * pi * np.sin(pi * x) * np.cos(pi * y)  # ∂u₁/∂y
    grad_u[1, 0] = -growth_factor_2 * pi * np.sin(pi * x) * np.cos(pi * y)  # ∂u₂/∂x
    grad_u[1, 1] = -growth_factor_2 * pi * np.cos(pi * x) * np.sin(pi * y)  # ∂u₂/∂y
    lap_u = np.array([-2 * pi**2 * u_vals[0], -2 * pi**2 * u_vals[1]])  
    return u_t, grad_u, lap_u


def precompute_source_terms(times, interior_points, W_u_all, grad_W_all, velocity_field, velocity_jacobian, nu):
    print("   Precomputing source terms...")
    start_time = time()    
    n_times = len(times)
    n_points = len(interior_points)
    F_all = np.zeros((n_times, n_points, 2))
    nu_np = np.array(nu)
    
    for t_idx, t in enumerate(times):
        for i, point in enumerate(interior_points):
            x, y = point[0], point[1]
            u_exact_val = u_exact_general(t, x, y)
            u_t, grad_u, lap_u = u_exact_derivatives_general(t, x, y)
            W_u = W_u_all[t_idx, i, :]
            grad_W = grad_W_all[t_idx, i, :, :]  # shape (2, 2)

            V = velocity_field(W_u)
            JV = velocity_jacobian(W_u)
            
            DV = np.einsum('dc,cd->', JV, grad_W)
            adv_term = np.einsum('i,pi->p', V, grad_u)
            N_u = nonlinear_function(u_exact_val)
            F_all[t_idx, i, :] = u_t + DV * u_exact_val + adv_term - nu_np @ lap_u - N_u
    
    print(f"   ...source terms computed in {time() - start_time:.2f}s")
    return F_all

In [ ]:
def compute_rhs(t, X, G_tilde, F, precomputed, velocity_field, velocity_jacobian, nu):
    n, p = X.shape
    Ka, Kb = precomputed['Ka'], precomputed['Kb']
    dKa_dx, dKa_dy = precomputed['dKa_dx'], precomputed['dKa_dy']
    dKb_dx, dKb_dy = precomputed['dKb_dx'], precomputed['dKb_dy']
    La, Lb = precomputed['La'], precomputed['Lb']
    grad_a, grad_b = precomputed['grad_a'], precomputed['grad_b']
    
    Wu = Ka @ X + Kb @ G_tilde
    grad_Wu_x = dKa_dx @ X + dKb_dx @ G_tilde
    grad_Wu_y = dKa_dy @ X + dKb_dy @ G_tilde
    grad_Wu = np.stack([grad_Wu_x, grad_Wu_y], axis=2)
    
    V_all = np.array([velocity_field(Wu[i, :]) for i in range(n)])
    JV_all = np.array([velocity_jacobian(Wu[i, :]) for i in range(n)])
    
    DV = np.einsum('idc,icd->i', JV_all, grad_Wu)
    grad_u = np.einsum('ikd,kj->ijd', grad_a, X) + np.einsum('ild,lj->ijd', grad_b, G_tilde)
    adv_term = np.einsum('id,ipd->ip', V_all, grad_u)
    
    lap_u = La @ X + Lb @ G_tilde  
    nu_np = np.array(nu)
    diffusion_term = lap_u @ nu_np.T  
    u_interior = X + precomputed['Kb'] @ G_tilde / (precomputed['Ka'].diagonal()[:, np.newaxis] + 1e-10)
    N_u = np.array([nonlinear_function(X[i, :]) for i in range(n)])
    dX_dt = F - (DV[:, np.newaxis] * X + adv_term - diffusion_term - N_u)
    return dX_dt

def frechet_derivative(t, X, G_tilde, H, precomputed, velocity_field, velocity_jacobian, velocity_hessian, nu):
    n, p = X.shape
    Ka, Kb = precomputed['Ka'], precomputed['Kb']
    dKa_dx, dKa_dy = precomputed['dKa_dx'], precomputed['dKa_dy']
    dKb_dx, dKb_dy = precomputed['dKb_dx'], precomputed['dKb_dy']
    La, Lb = precomputed['La'], precomputed['Lb']
    grad_a, grad_b = precomputed['grad_a'], precomputed['grad_b']
    Wu = Ka @ X + Kb @ G_tilde
    grad_Wu = np.stack([dKa_dx @ X + dKb_dx @ G_tilde, dKa_dy @ X + dKb_dy @ G_tilde], axis=2)
    grad_u = np.einsum('ikd,kj->ijd', grad_a, X) + np.einsum('ild,lj->ijd', grad_b, G_tilde)    
    V_all = np.array([velocity_field(Wu[i, :]) for i in range(n)])
    JV_all = np.array([velocity_jacobian(Wu[i, :]) for i in range(n)])
    HV_all = np.array([velocity_hessian(Wu[i, :]) for i in range(n)])    
    DV = np.einsum('idc,icd->i', JV_all, grad_Wu)    
    delta_Wu = Ka @ H
    delta_grad_Wu = np.stack([dKa_dx @ H, dKa_dy @ H], axis=2)    
    delta_DV_hess = np.einsum('idco,io,icd->i', HV_all, delta_Wu, grad_Wu)
    delta_DV_jac = np.einsum('idc,icd->i', JV_all, delta_grad_Wu)
    delta_DV = delta_DV_hess + delta_DV_jac    
    delta_V = np.einsum('ipo,io->ip', JV_all, delta_Wu)
    grad_H = np.einsum('ikd,kj->ijd', grad_a, H)
    term1 = DV[:, np.newaxis] * H
    term2 = np.einsum('id,ipd->ip', V_all, grad_H)
    nu_np = np.array(nu)
    term3 = -(La @ H) @ nu_np.T  
    term4 = delta_DV[:, np.newaxis] * X
    term5 = np.einsum('id,ipd->ip', delta_V, grad_u)
    

    JN_term = np.zeros((n, p))
    for i in range(n):
        JN_i = nonlinear_jacobian(X[i, :])
        JN_term[i, :] = JN_i @ H[i, :]
    return -(term1 + term2 + term3 + term4 + term5 - JN_term)

In [ ]:
BDF_COEFFS = {
    1: {'beta': 1.0, 'eta': [1.0]},
    2: {'beta': 2/3, 'eta': [4/3, -1/3]},
    3: {'beta': 6/11, 'eta': [18/11, -9/11, 2/11]},
    4: {'beta': 12/25, 'eta': [48/25, -36/25, 16/25, -3/25]},
    5: {'beta': 60/137, 'eta': [300/137, -300/137, 200/137, -75/137, 12/137]},
    6: {'beta': 60/147, 'eta': [360/147, -450/147, 400/147, -225/147, 72/147, -10/147]}
}

def newton_krylov_solve(t, dt, beta, eta_sum, G_tilde, X_guess, F_current,rhs_func, frechet_func, preconditioner_op=None,newton_tol=1e-7, 
                        newton_max_iter=20, gmres_tol=1e-7, verbose=False):
    X_current = X_guess.copy()
    n, p = X_current.shape
    size = n * p 
    R_current = rhs_func(t, X_current, G_tilde, F_current)
    F_current_residual = X_current - beta * dt * R_current - eta_sum
    for newton_iter in range(newton_max_iter):
        F_norm = norm(F_current_residual.flatten())
        
        if verbose and newton_iter < 10:
            print(f"    Newton iter {newton_iter}: ||F|| = {F_norm:.2e}")
        
        if F_norm < newton_tol:
            if verbose: print(f"    Converged in {newton_iter} iterations.")
            return X_current, True
        
        def matvec(v):
            v_shaped = v.reshape(n, p)
            DR_v = frechet_func(t, X_current, G_tilde, v_shaped)
            result = v_shaped - beta * dt * DR_v
            return result.flatten()
        J_op = LinearOperator((size, size), matvec=matvec)
        delta_X_flat, info = gmres(J_op, -F_current_residual.flatten(), rtol=gmres_tol, maxiter=150, M=preconditioner_op)
        if info != 0 and verbose:
            print(f"    WARNING: GMRES info={info}. Update may be inaccurate.")           
        delta_X = delta_X_flat.reshape(n, p)    
        alpha = 1.0
        for _ in range(5):
            X_new = X_current + alpha * delta_X
            R_new = rhs_func(t, X_new, G_tilde, F_current)
            F_new_residual = X_new - beta * dt * R_new - eta_sum
            F_norm_new = norm(F_new_residual.flatten())           
            if F_norm_new < F_norm:
                break
            alpha /= 2.0
        X_current = X_new
        F_current_residual = F_new_residual      
    if verbose: print(f"    WARNING: Newton did not converge in {newton_max_iter} iterations.")
    return X_current, False

In [ ]:
print("=" * 70)
print("Running Test on Flower Domain with Precomputed Manufactured Solution")
print("=" * 70)
n_interior = 50
n_boundary = 50
nu = [[0.05, 0],
     [0, 0.05]]  
dt = 0.05
t_final = 500
p = 2
alpha = 2.0
polydeg = 4
tau = 5
print(f"\nParameters: n_int={n_interior}, n_bnd={n_boundary}, nu={nu}, dt={dt}, T_final={t_final}")
print(f"            Using Halton sequence for quasi-random interior points")
print("\n[1] Setting up grid and RBF matrices...")
start_time = time()
interior_points, boundary_points = generate_flower_domain_points(n_interior, n_boundary)
matrices = build_matrices(interior_points, boundary_points, tau, polydeg)
La, Lb = compute_basis_laplacians(interior_points, boundary_points, matrices, tau, polydeg)
grad_a, grad_b = compute_basis_gradients(interior_points, interior_points, boundary_points, matrices, tau, polydeg)
print(f"   ...setup completed in {time() - start_time:.2f}s")
print("\n[2] Computing nonlocal operators (integral over FLOWER domain)...")
def setup_quadrature(order=30, a_int=0.0, b_int=1.0):
    xi_1d, w_1d = leggauss(order)
    L = b_int - a_int
    quad_points_1d = a_int + 0.5 * L * (xi_1d + 1)
    quad_weights_1d = 0.5 * L * w_1d
    XX, YY = np.meshgrid(quad_points_1d, quad_points_1d)
    WX, WY = np.meshgrid(quad_weights_1d, quad_weights_1d)
    return np.column_stack([XX.ravel(), YY.ravel()]), (WX * WY).ravel()
start_time = time()
quad_points, quad_weights = setup_quadrature(order=40, a_int=0.0, b_int=1.0)
domain_checker_flower = lambda x, y: is_point_in_flower(x, y)
Ka, Kb = compute_Ka_Kb_matrices(interior_points, boundary_points, matrices, quad_points, quad_weights, domain_checker_flower, tau, alpha, polydeg)
dKa_dx, dKa_dy, dKb_dx, dKb_dy = compute_Ka_Kb_gradients(interior_points, boundary_points, matrices, quad_points, quad_weights,
                                                         domain_checker_flower, tau, alpha, polydeg)
print(f"   ...nonlocal operators computed in {time() - start_time:.2f}s")
def velocity_field(Wu):
    V = np.zeros_like(Wu)
    V[0] = 0.5 * np.exp(-1.5 * Wu[0]) * (1 - 0.3 * Wu[1])
    V[1] = 0.4 * np.exp(-1.8 * Wu[1]) * (1 - 0.25 * Wu[0])
    return V

def velocity_jacobian(Wu):
    exp_0 = np.exp(-1.5 * Wu[0])
    exp_1 = np.exp(-1.8 * Wu[1])
    factor_0 = (1 - 0.3 * Wu[1])
    factor_1 = (1 - 0.25 * Wu[0])
    J_00 = -0.75 * exp_0 * factor_0
    J_01 = -0.15 * exp_0
    J_10 = -0.1 * exp_1
    J_11 = -0.72 * exp_1 * factor_1
    
    return np.array([
        [J_00, J_01],
        [J_10, J_11]
    ])

def velocity_hessian(Wu):
    H = np.zeros((2, 2, 2))
    exp_0 = np.exp(-1.5 * Wu[0])
    exp_1 = np.exp(-1.8 * Wu[1])
    factor_0 = (1 - 0.3 * Wu[1])
    factor_1 = (1 - 0.25 * Wu[0])
    H[0, 0, 0] = 1.125 * exp_0 * factor_0
    H[0, 1, 1] = 0.0
    H[1, 0, 0] = 0.0
    H[1, 1, 1] = 1.296 * exp_1 * factor_1
    return H

    
def nonlinear_function(u):
    N = np.zeros_like(u)
    N[0] = 0.15 * np.tanh(u[0]) - 0.05 * np.tanh(u[1]**2)
    N[1] = 0.1 * np.tanh(u[1]) + 0.05 * u[0] * np.exp(-u[0]**2)
    return N

def nonlinear_jacobian(u):
    JN = np.zeros((2, 2))
    JN[0, 0] = 0.15 * (1 - np.tanh(u[0])**2)
    JN[0, 1] = -0.05 * 2 * u[1] * (1 - np.tanh(u[1]**2)**2)
    JN[1, 0] = 0.05 * np.exp(-u[0]**2) * (1 - 2*u[0]**2)
    JN[1, 1] = 0.1 * (1 - np.tanh(u[1])**2)
    return JN    
print("\n[3] Precomputing manufactured solution and source terms...")
n_steps = int(t_final / dt)
times = np.linspace(0, t_final, n_steps + 1)
W_u_all, grad_W_all = precompute_manufactured_solution(times, interior_points, alpha)
F_all = precompute_source_terms(times, interior_points, W_u_all, grad_W_all, velocity_field, velocity_jacobian, nu)
print("\n[4] Building the ILU Preconditioner...")
start_time = time()
n = matrices['n']
I_n = identity(n, format='csc')
nu_np = np.array(nu)
J_approx_component_1 = I_n - dt * nu_np[0, 0] * csc_matrix(La)
J_approx_component_2 = I_n - dt * nu_np[1, 1] * csc_matrix(La)


J_approx_sparse = block_diag([J_approx_component_1, J_approx_component_2], format='csc')
ilu = spilu(J_approx_sparse)
preconditioner_op = LinearOperator(J_approx_sparse.shape, ilu.solve)
print(f"   ...preconditioner built in {time() - start_time:.2f}s")
print("\n[5] Starting BDF time integration...")
initial_condition = np.array([u_exact_general(0, pt[0], pt[1]).T for pt in interior_points])
dm = (polydeg + 1) * (polydeg + 2) // 2

def boundary_condition(t):
    g_vals = np.array([u_exact_general(t, pt[0], pt[1]).T for pt in boundary_points])
    return np.vstack([g_vals, np.zeros((dm, p))])
precomputed = {
    'Ka': Ka, 'Kb': Kb, 
    'dKa_dx': dKa_dx, 'dKa_dy': dKa_dy, 
    'dKb_dx': dKb_dx, 'dKb_dy': dKb_dy, 
    'La': La, 'Lb': Lb, 
    'grad_a': grad_a, 'grad_b': grad_b, 
    'interior_points': interior_points
}

rhs_func = lambda t, X, G, F: compute_rhs(t, X, G, F, precomputed, velocity_field, velocity_jacobian, nu)
frechet_func = lambda t, X, G, H: frechet_derivative(t, X, G, H, precomputed, velocity_field, velocity_jacobian, velocity_hessian, nu)

X_history = [initial_condition]

integration_start_time = time()
for step in range(1, n_steps + 1):
    t = times[step]
    order = min(step, 6)
    coeffs = BDF_COEFFS[order]
    beta, eta = coeffs['beta'], coeffs['eta']

    eta_sum = np.zeros_like(initial_condition)
    for j, eta_j in enumerate(eta):
        eta_sum += eta_j * X_history[step - j - 1]
    
    G_tilde = boundary_condition(t)
    X_guess = X_history[-1]

    F_current = F_all[step]
    
    X_new, converged = newton_krylov_solve(t, dt, beta, eta_sum, G_tilde, X_guess, F_current, rhs_func, frechet_func,
                                           preconditioner_op=preconditioner_op)
    X_history.append(X_new)
    
    if step % 10 == 0:
        print(f"   Step {step}/{n_steps} completed (t={t:.3f})")
            
print(f"   ...integration completed in {time() - integration_start_time:.2f}s")
X_history = np.array(X_history)
    

In [ ]:
print("\n[6] Analyzing and visualizing results...")

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Polygon
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np

# Set publication-quality defaults
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1

import os
if not os.path.exists('figures'):
    os.makedirs('figures')


fig1 = plt.figure(figsize=(6, 6))
ax1 = fig1.add_subplot(111)
theta_plot = np.linspace(0, 2 * np.pi, 200)
r_plot = flower_radius(theta_plot)
ax1.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
         'k-', linewidth=1.5, label='Domain Boundary')
ax1.scatter(interior_points[:, 0], interior_points[:, 1], c='darkred', marker='x', 
           s=20, label='Interior Points', alpha=0.7)
ax1.scatter(boundary_points[:, 0], boundary_points[:, 1], c='darkblue', marker='o', 
           facecolors='none', s=25, label='Boundary Points', linewidths=1)
ax1.set_title('Collocation Points Distribution')
ax1.set_xlabel('$x$')
ax1.set_ylabel('$y$')
ax1.legend(loc='best', frameon=True, fancybox=False, shadow=False)
ax1.axis('equal')
ax1.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/collocation_points.pdf', format='pdf')
plt.savefig('figures/collocation_points.png', format='png')
plt.show()


errors_rel_Linf = []
errors_rel_L2 = []
errors_abs_Linf = []
errors_abs_L2 = []

for k, t in enumerate(times):
    exact_sol = np.array([u_exact_general(t, pt[0], pt[1]).T 
                          for pt in interior_points])
    error_vec = X_history[k] - exact_sol
    errors_abs_Linf.append(np.max(np.abs(error_vec), axis=0))
    errors_abs_L2.append(np.sqrt(np.mean(error_vec**2, axis=0)))
    
    exact_norm_Linf = np.max(np.abs(exact_sol), axis=0)
    exact_norm_L2 = np.sqrt(np.mean(exact_sol**2, axis=0))
    
    rel_err_Linf = np.max(np.abs(error_vec), axis=0) / (exact_norm_Linf + 1e-16)
    rel_err_L2 = np.sqrt(np.mean(error_vec**2, axis=0)) / (exact_norm_L2 + 1e-16)
    
    errors_rel_Linf.append(rel_err_Linf)
    errors_rel_L2.append(rel_err_L2)

errors_rel_Linf = np.array(errors_rel_Linf)
errors_rel_L2 = np.array(errors_rel_L2)
errors_abs_Linf = np.array(errors_abs_Linf)
errors_abs_L2 = np.array(errors_abs_L2)


fig2 = plt.figure(figsize=(7, 5))
ax2 = fig2.add_subplot(111)
ax2.semilogy(times, errors_rel_Linf[:, 0], 'b-', linewidth=1.5, label='$u_1$ ($L_\infty$)')
ax2.semilogy(times, errors_rel_Linf[:, 1], 'r-', linewidth=1.5, label='$u_2$ ($L_\infty$)')
ax2.semilogy(times, errors_rel_L2[:, 0], 'b--', linewidth=1.5, label='$u_1$ ($L_2$)', alpha=0.7)
ax2.semilogy(times, errors_rel_L2[:, 1], 'r--', linewidth=1.5, label='$u_2$ ($L_2$)', alpha=0.7)
ax2.set_xlabel('Time $t$')
ax2.set_ylabel('Relative Error')
ax2.set_title('Temporal Evolution of Relative Errors')
ax2.legend(loc='best', ncol=2, frameon=True, fancybox=False)
ax2.grid(True, which="both", ls="--", alpha=0.3, linewidth=0.5)
ax2.set_xlim([times[0], times[-1]])
plt.tight_layout()
plt.savefig('figures/error_evolution.pdf', format='pdf')
plt.savefig('figures/error_evolution.png', format='png')
plt.show()
def numerical_sol(eval_points, X_values, G_tilde, interior_points, boundary_points, matrices, tau, polydeg):
    a_mat, b_mat = evaluate_basis_functions(eval_points, interior_points, boundary_points, matrices, tau, polydeg)
    u_h = a_mat @ X_values + b_mat @ G_tilde
    return u_h


t_plot_idx = -1
t_plot = times[t_plot_idx]
print(f"\nGenerating solution fields at t={t_plot:.3f} using proper RBF interpolation...")

resolution = 1000  # keep your resolution
grid_x, grid_y = np.mgrid[0:1:complex(0, resolution), 0:1:complex(0, resolution)]
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])

X_final = X_history[t_plot_idx]                # (n_interior, 2)
G_final = boundary_condition(t_plot)           # (n_boundary + dm, 2)

# numerical via basis (FIXED variable names)
print("  Evaluating RBF solution on grid...")
u_num_flat =numerical_sol(grid_points, X_final, G_final,
                                           interior_points, boundary_points,
                                           matrices, tau, polydeg)  # (N,2)
if u_num_flat.ndim != 2 or u_num_flat.shape[1] != 2:
    raise ValueError(f"u_num_flat has wrong shape {u_num_flat.shape}, expected (N, 2).")

u_num_comp1 = u_num_flat[:, 0].reshape(grid_x.shape)
u_num_comp2 = u_num_flat[:, 1].reshape(grid_x.shape)


print("  Computing exact solution on grid...")
u_exact_grid = u_exact_general(t_plot, grid_x, grid_y)  

mask = ~is_point_in_flower(grid_x, grid_y)
u_num_comp1[mask] = np.nan
u_num_comp2[mask] = np.nan
u_exact_grid[0][mask] = np.nan
u_exact_grid[1][mask] = np.nan


fig3 = plt.figure(figsize=(5.5, 5))
ax3 = fig3.add_subplot(111)
levels = np.linspace(np.nanmin(u_num_comp1), np.nanmax(u_num_comp1), 20)
c3 = ax3.contourf(grid_x, grid_y, u_num_comp1, levels=levels, cmap='viridis')
ax3.contour(grid_x, grid_y, u_num_comp1, levels=levels[::4], colors='black', alpha=0.3, linewidths=0.5)
# Add domain boundary
ax3.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
         'k-', linewidth=1.5)
ax3.set_title(f'$u_1$ (Numerical) at $t={int(round(t_plot))}$')
ax3.set_xlabel('$x$')
ax3.set_ylabel('$y$')
cbar = plt.colorbar(c3, ax=ax3, label='$u_1$')
cbar.ax.tick_params(labelsize=9)
ax3.axis('equal')
ax3.set_xlim(-0.05, 1.05)
ax3.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/u1_numerical_contour.pdf', format='pdf')
plt.savefig('figures/u1_numerical_contour.png', format='png')
plt.show()


fig4 = plt.figure(figsize=(5.5, 5))
ax4 = fig4.add_subplot(111)
levels = np.linspace(np.nanmin(u_exact_grid[0]), np.nanmax(u_exact_grid[0]), 20)
c4 = ax4.contourf(grid_x, grid_y, u_exact_grid[0], levels=levels, cmap='viridis')
ax4.contour(grid_x, grid_y, u_exact_grid[0], levels=levels[::4], colors='black', alpha=0.3, linewidths=0.5)
ax4.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
         'k-', linewidth=1.5)
ax4.set_title(f'$u_1$ (Exact) at $t={int(round(t_plot))}$')
ax4.set_xlabel('$x$')
ax4.set_ylabel('$y$')
cbar = plt.colorbar(c4, ax=ax4, label='$u_1$')
cbar.ax.tick_params(labelsize=9)
ax4.axis('equal')
ax4.set_xlim(-0.05, 1.05)
ax4.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/u1_exact_contour.pdf', format='pdf')
plt.savefig('figures/u1_exact_contour.png', format='png')
plt.show()


fig5 = plt.figure(figsize=(5.5, 5))
ax5 = fig5.add_subplot(111)
levels = np.linspace(np.nanmin(u_num_comp2), np.nanmax(u_num_comp2), 20)
c5 = ax5.contourf(grid_x, grid_y, u_num_comp2, levels=levels, cmap='plasma')
ax5.contour(grid_x, grid_y, u_num_comp2, levels=levels[::4], colors='black', alpha=0.3, linewidths=0.5)
ax5.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
         'k-', linewidth=1.5)
ax5.set_title(f'$u_2$ (Numerical) at $t={int(round(t_plot))}$')
ax5.set_xlabel('$x$')
ax5.set_ylabel('$y$')
cbar = plt.colorbar(c5, ax=ax5, label='$u_2$')
cbar.ax.tick_params(labelsize=9)
ax5.axis('equal')
ax5.set_xlim(-0.05, 1.05)
ax5.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/u2_numerical_contour.pdf', format='pdf')
plt.savefig('figures/u2_numerical_contour.png', format='png')
plt.show()


fig6 = plt.figure(figsize=(5.5, 5))
ax6 = fig6.add_subplot(111)
levels = np.linspace(np.nanmin(u_exact_grid[1]), np.nanmax(u_exact_grid[1]), 20)
c6 = ax6.contourf(grid_x, grid_y, u_exact_grid[1], levels=levels, cmap='plasma')
ax6.contour(grid_x, grid_y, u_exact_grid[1], levels=levels[::4], colors='black', alpha=0.3, linewidths=0.5)
ax6.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
         'k-', linewidth=1.5)
ax6.set_title(f'$u_2$ (Exact) at $t={int(round(t_plot))}$')
ax6.set_xlabel('$x$')
ax6.set_ylabel('$y$')
cbar = plt.colorbar(c6, ax=ax6, label='$u_2$')
cbar.ax.tick_params(labelsize=9)
ax6.axis('equal')
ax6.set_xlim(-0.05, 1.05)
ax6.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/u2_exact_contour.pdf', format='pdf')
plt.savefig('figures/u2_exact_contour.png', format='png')
plt.show()


fig_err1 = plt.figure(figsize=(5.5, 5))
ax_err1 = fig_err1.add_subplot(111)
rel_error_comp1 = np.abs(u_num_comp1 - u_exact_grid[0]) / (np.abs(u_exact_grid[0]) + 1e-10)
rel_error_comp1[mask] = np.nan
levels_err = np.logspace(-6, 0, 20)
c_err1 = ax_err1.contourf(grid_x, grid_y, rel_error_comp1, levels=levels_err, 
                           cmap='hot', norm=plt.matplotlib.colors.LogNorm())
ax_err1.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
             'k-', linewidth=1.5)
ax_err1.set_title(f'Relative Error in $u_1$ at $t={int(round(t_plot))}$')
ax_err1.set_xlabel('$x$')
ax_err1.set_ylabel('$y$')
cbar = plt.colorbar(c_err1, ax=ax_err1, label='Relative Error')
cbar.ax.tick_params(labelsize=9)
ax_err1.axis('equal')
ax_err1.set_xlim(-0.05, 1.05)
ax_err1.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/u1_relative_error.pdf', format='pdf')
plt.savefig('figures/u1_relative_error.png', format='png')
plt.show()


fig_err2 = plt.figure(figsize=(5.5, 5))
ax_err2 = fig_err2.add_subplot(111)
rel_error_comp2 = np.abs(u_num_comp2 - u_exact_grid[1]) / (np.abs(u_exact_grid[1]) + 1e-10)
rel_error_comp2[mask] = np.nan
c_err2 = ax_err2.contourf(grid_x, grid_y, rel_error_comp2, levels=levels_err, 
                           cmap='hot', norm=plt.matplotlib.colors.LogNorm())
ax_err2.plot(0.5 + r_plot * np.cos(theta_plot), 0.5 + r_plot * np.sin(theta_plot), 
             'k-', linewidth=1.5)
ax_err2.set_title(f'Relative Error in $u_2$ at $t={int(round(t_plot))}$')
ax_err2.set_xlabel('$x$')
ax_err2.set_ylabel('$y$')
cbar = plt.colorbar(c_err2, ax=ax_err2, label='Relative Error')
cbar.ax.tick_params(labelsize=9)
ax_err2.axis('equal')
ax_err2.set_xlim(-0.05, 1.05)
ax_err2.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('figures/u2_relative_error.pdf', format='pdf')
plt.savefig('figures/u2_relative_error.png', format='png')
plt.show()


fig7 = plt.figure(figsize=(8, 6))
ax7 = fig7.add_subplot(111, projection='3d')


u_num_comp1_masked = np.ma.masked_invalid(u_num_comp1)


surf7 = ax7.plot_surface(grid_x, grid_y, u_num_comp1_masked, 
                         cmap='viridis', edgecolor='none', alpha=0.95,
                         linewidth=0, antialiased=True, shade=True,
                         vmin=np.nanmin(u_num_comp1), vmax=np.nanmax(u_num_comp1))


z_min = np.nanmin(u_num_comp1_masked) - 0.1 * (np.nanmax(u_num_comp1_masked) - np.nanmin(u_num_comp1_masked))
shadow = np.ones_like(grid_x) * z_min
shadow[mask] = np.nan
ax7.contourf(grid_x, grid_y, shadow, zdir='z', offset=z_min, 
             colors='gray', alpha=0.3)


boundary_x = 0.5 + r_plot * np.cos(theta_plot)
boundary_y = 0.5 + r_plot * np.sin(theta_plot)
boundary_z = np.ones_like(boundary_x) * z_min
ax7.plot(boundary_x, boundary_y, boundary_z, 'k-', linewidth=2, alpha=0.8)

ax7.set_title(f'$u_1$ (Numerical) at $t={t_plot:.3f}$')
ax7.set_xlabel('$x$')
ax7.set_ylabel('$y$')
ax7.set_zlabel('$u_1$')
ax7.view_init(elev=25, azim=45)
ax7.set_box_aspect([1, 1, 0.5])
cbar = fig7.colorbar(surf7, ax=ax7, shrink=0.6, aspect=10, pad=0.1)
cbar.ax.tick_params(labelsize=9)
plt.tight_layout()
plt.savefig('figures/u1_numerical_3d.pdf', format='pdf')
plt.savefig('figures/u1_numerical_3d.png', format='png')
plt.show()


fig8 = plt.figure(figsize=(8, 6))
ax8 = fig8.add_subplot(111, projection='3d')

u_exact_comp1_masked = np.ma.masked_invalid(u_exact_grid[0])

surf8 = ax8.plot_surface(grid_x, grid_y, u_exact_comp1_masked, 
                         cmap='viridis', edgecolor='none', alpha=0.95,
                         linewidth=0, antialiased=True, shade=True,
                         vmin=np.nanmin(u_exact_grid[0]), vmax=np.nanmax(u_exact_grid[0]))

z_min = np.nanmin(u_exact_comp1_masked) - 0.1 * (np.nanmax(u_exact_comp1_masked) - np.nanmin(u_exact_comp1_masked))
shadow = np.ones_like(grid_x) * z_min
shadow[mask] = np.nan
ax8.contourf(grid_x, grid_y, shadow, zdir='z', offset=z_min, 
             colors='gray', alpha=0.3)

boundary_z = np.ones_like(boundary_x) * z_min
ax8.plot(boundary_x, boundary_y, boundary_z, 'k-', linewidth=2, alpha=0.8)

ax8.set_title(f'$u_1$ (Exact) at $t={int(round(t_plot))}$')
ax8.set_xlabel('$x$')
ax8.set_ylabel('$y$')
ax8.set_zlabel('$u_1$')
ax8.view_init(elev=25, azim=45)
ax8.set_box_aspect([1, 1, 0.5])
cbar = fig8.colorbar(surf8, ax=ax8, shrink=0.6, aspect=10, pad=0.1)
cbar.ax.tick_params(labelsize=9)
plt.tight_layout()
plt.savefig('figures/u1_exact_3d.pdf', format='pdf')
plt.savefig('figures/u1_exact_3d.png', format='png')
plt.show()


fig9 = plt.figure(figsize=(8, 6))
ax9 = fig9.add_subplot(111, projection='3d')

u_num_comp2_masked = np.ma.masked_invalid(u_num_comp2)

surf9 = ax9.plot_surface(grid_x, grid_y, u_num_comp2_masked, 
                         cmap='plasma', edgecolor='none', alpha=0.95,
                         linewidth=0, antialiased=True, shade=True,
                         vmin=np.nanmin(u_num_comp2), vmax=np.nanmax(u_num_comp2))

z_min = np.nanmin(u_num_comp2_masked) - 0.1 * (np.nanmax(u_num_comp2_masked) - np.nanmin(u_num_comp2_masked))
shadow = np.ones_like(grid_x) * z_min
shadow[mask] = np.nan
ax9.contourf(grid_x, grid_y, shadow, zdir='z', offset=z_min, 
             colors='gray', alpha=0.3)

boundary_z = np.ones_like(boundary_x) * z_min
ax9.plot(boundary_x, boundary_y, boundary_z, 'k-', linewidth=2, alpha=0.8)

ax9.set_title(f'$u_2$ (Numerical) at $t={int(round(t_plot))}$')
ax9.set_xlabel('$x$')
ax9.set_ylabel('$y$')
ax9.set_zlabel('$u_2$')
ax9.view_init(elev=25, azim=45)
ax9.set_box_aspect([1, 1, 0.5])
cbar = fig9.colorbar(surf9, ax=ax9, shrink=0.6, aspect=10, pad=0.1)
cbar.ax.tick_params(labelsize=9)
plt.tight_layout()
plt.savefig('figures/u2_numerical_3d.pdf', format='pdf')
plt.savefig('figures/u2_numerical_3d.png', format='png')
plt.show()


fig10 = plt.figure(figsize=(8, 6))
ax10 = fig10.add_subplot(111, projection='3d')

u_exact_comp2_masked = np.ma.masked_invalid(u_exact_grid[1])

surf10 = ax10.plot_surface(grid_x, grid_y, u_exact_comp2_masked, 
                           cmap='plasma', edgecolor='none', alpha=0.95,
                           linewidth=0, antialiased=True, shade=True,
                           vmin=np.nanmin(u_exact_grid[1]), vmax=np.nanmax(u_exact_grid[1]))

z_min = np.nanmin(u_exact_comp2_masked) - 0.1 * (np.nanmax(u_exact_comp2_masked) - np.nanmin(u_exact_comp2_masked))
shadow = np.ones_like(grid_x) * z_min
shadow[mask] = np.nan
ax10.contourf(grid_x, grid_y, shadow, zdir='z', offset=z_min, 
              colors='gray', alpha=0.3)

boundary_z = np.ones_like(boundary_x) * z_min
ax10.plot(boundary_x, boundary_y, boundary_z, 'k-', linewidth=2, alpha=0.8)

ax10.set_title(f'$u_2$ (Exact) at $t={int(round(t_plot))}$')
ax10.set_xlabel('$x$')
ax10.set_ylabel('$y$')
ax10.set_zlabel('$u_2$')
ax10.view_init(elev=25, azim=45)
ax10.set_box_aspect([1, 1, 0.5])
cbar = fig10.colorbar(surf10, ax=ax10, shrink=0.6, aspect=10, pad=0.1)
cbar.ax.tick_params(labelsize=9)
plt.tight_layout()
plt.savefig('figures/u2_exact_3d.pdf', format='pdf')
plt.savefig('figures/u2_exact_3d.png', format='png')
plt.show()

print("\n" + "=" * 70)
print("FINAL STATISTICS")
print("=" * 70)
print(f"Final time: {t_final}")
print(f"\nRelative Errors:")
print(f"  L∞: Component 1 = {errors_rel_Linf[-1, 0]:.2e}, Component 2 = {errors_rel_Linf[-1, 1]:.2e}")
print(f"  L2: Component 1 = {errors_rel_L2[-1, 0]:.2e}, Component 2 = {errors_rel_L2[-1, 1]:.2e}")
print(f"\nAbsolute Errors:")
print(f"  L∞: Component 1 = {errors_abs_Linf[-1, 0]:.2e}, Component 2 = {errors_abs_Linf[-1, 1]:.2e}")
print(f"  L2: Component 1 = {errors_abs_L2[-1, 0]:.2e}, Component 2 = {errors_abs_L2[-1, 1]:.2e}")
print(f"\nMax relative error over all time: {np.max(errors_rel_Linf):.2e}")
print(f"Max absolute error over all time: {np.max(errors_abs_Linf):.2e}")

np.savez('error_analysis.npz',
         times=times,
         rel_Linf=errors_rel_Linf,
         rel_L2=errors_rel_L2,
         abs_Linf=errors_abs_Linf,
         abs_L2=errors_abs_L2)

print("\nAll figures saved to 'figures/' directory in PDF and PNG formats.")